# Stage 2 controlled Adam baseline

This notebook runs **Adam only** for the 2-D Poisson PINN using the already-created frozen batch schedule. Run it in Kaggle and change only `steps` and `output_name` for the 500-step and 2000-step runs.

The matching SOAP experiment is in `soap.ipynb`.

## Kaggle setup

1. Open this notebook in Kaggle and enable **Settings > Accelerator > GPU**.
2. Upload `poisson_seed0_K1_schedule.pt` from the earlier controlled Stage-2 run, or place it in `/kaggle/working/results/`.
3. Run every cell from top to bottom.
4. For the requested runs, use `steps = 500` with `stage2_adam_seed0_K1_500.json`, then `steps = 2000` with `stage2_adam_seed0_K1_2000.json`.

This notebook does not import or run SOAP.

In [ ]:
from __future__ import annotations

import json
import math
import time
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn

output_dir = Path('/kaggle/working/results')
output_dir.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch: {torch.__version__}')

## Frozen experiment configuration

Change only `steps` and `output_name` between the two requested runs. The notebook always loads the existing 2000-step schedule and never regenerates it.

In [ ]:
# ===== Run Adam for one requested duration =====
optimizer = 'adam'
steps = 500  # Change to 2000 for the second run.
schedule_steps = 2000
seed = 0
K = 1
n_pde = 1000
n_boundary_each = 100
lr = 1e-3
eval_every = 100
schedule_path = output_dir / f'poisson_seed{seed}_K{K}_schedule.pt'
output_name = f'stage2_adam_seed{seed}_K{K}_{steps}.json'

print({
    'optimizer': optimizer,
    'steps': steps,
    'schedule_steps': schedule_steps,
    'seed': seed,
    'K': K,
    'n_pde': n_pde,
    'n_boundary_each': n_boundary_each,
    'lr': lr,
    'eval_every': eval_every,
    'schedule': str(schedule_path),
    'output': str(output_dir / output_name),
})

### Frozen schedule handling

If `poisson_seed0_K1_schedule.pt` is already in `/kaggle/working/results/`, this notebook loads it unchanged. Otherwise, Adam creates the deterministic 2000-step schedule with seed 0 and saves it there. The SOAP notebook must use this saved file for the matching comparison.

In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)


def set_seed(seed_value: int) -> None:
    torch.manual_seed(seed_value)
    np.random.seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)


def sample_boundary(n_each: int) -> torch.Tensor:
    values = torch.rand(4 * n_each, device=device)
    boundary = torch.empty(4 * n_each, 2, device=device)
    boundary[:n_each] = torch.stack([torch.zeros(n_each, device=device), values[:n_each]], dim=1)
    boundary[n_each:2 * n_each] = torch.stack([torch.ones(n_each, device=device), values[n_each:2 * n_each]], dim=1)
    boundary[2 * n_each:3 * n_each] = torch.stack([values[2 * n_each:3 * n_each], torch.zeros(n_each, device=device)], dim=1)
    boundary[3 * n_each:] = torch.stack([values[3 * n_each:], torch.ones(n_each, device=device)], dim=1)
    return boundary


def make_batch_schedule(schedule_length: int) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    for step in range(1, schedule_length + 1):
        pde = torch.rand(n_pde, 2, device=device)
        boundary = sample_boundary(n_boundary_each)
        schedule.append({'pde': pde.detach().clone(), 'bc': boundary.detach().clone()})
    return schedule


if schedule_path.exists():
    schedule = torch.load(schedule_path, map_location=device, weights_only=True)
    if len(schedule) < schedule_steps:
        raise ValueError(f'Schedule has {len(schedule)} entries; expected {schedule_steps}')
    print(f'Loaded frozen schedule: {schedule_path}')
else:
    set_seed(seed)
    schedule = make_batch_schedule(schedule_steps)
    torch.save(schedule, schedule_path)
    print(f'Created and saved frozen schedule: {schedule_path}')



def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    prediction = model(xy)
    first_derivatives = torch.autograd.grad(
        prediction, xy, grad_outputs=torch.ones_like(prediction), create_graph=True, retain_graph=True
    )[0]
    u_x = first_derivatives[:, 0:1]
    u_y = first_derivatives[:, 1:2]
    u_xx = torch.autograd.grad(
        u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True
    )[0][:, 0:1]
    u_yy = torch.autograd.grad(
        u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True
    )[0][:, 1:2]
    forcing = 2.0 * math.pi ** 2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing


def relative_l2(model: nn.Module) -> float:
    grid = torch.linspace(0.0, 1.0, 81, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing='ij')
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        prediction = model(xy).reshape(-1)
    reference = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(prediction - reference) / (torch.linalg.vector_norm(reference) + 1e-12))


def flat_grad(model: nn.Module) -> torch.Tensor:
    gradients = [parameter.grad.detach().reshape(-1) for parameter in model.parameters() if parameter.grad is not None]
    if not gradients:
        raise RuntimeError('No gradients available')
    return torch.cat(gradients)

In [ ]:
set_seed(seed)
model = PINN().to(device)
optimizer_instance = torch.optim.Adam(model.parameters(), lr=lr)

previous_gradient: Optional[torch.Tensor] = None
history: List[Dict[str, float]] = []
start_time = time.perf_counter()
model.train()

for step in range(1, steps + 1):
    current_pde = schedule[step - 1]['pde'].detach().clone().requires_grad_(True)
    current_boundary = schedule[step - 1]['bc']
    optimizer_instance.zero_grad(set_to_none=True)

    residual = poisson_residual(model, current_pde)
    loss_pde = torch.mean(residual.square())
    boundary_prediction = model(current_boundary)
    loss_boundary = torch.mean(boundary_prediction.square())
    loss = loss_pde + loss_boundary
    loss.backward()

    gradient = flat_grad(model)
    gradient_cosine = float('nan')
    if previous_gradient is not None:
        gradient_cosine = float(
            torch.dot(gradient, previous_gradient)
            / (torch.linalg.vector_norm(gradient) * torch.linalg.vector_norm(previous_gradient) + 1e-12)
        )
    gradient_norm = float(torch.linalg.vector_norm(gradient))
    previous_gradient = gradient.clone()
    parameters_before = torch.cat([parameter.detach().reshape(-1) for parameter in model.parameters()])
    optimizer_instance.step()
    parameters_after = torch.cat([parameter.detach().reshape(-1) for parameter in model.parameters()])
    update_norm = float(torch.linalg.vector_norm(parameters_after - parameters_before))

    if step == 1 or step % eval_every == 0 or step == steps:
        row = {
            'step': step,
            'loss': float(loss.detach()),
            'loss_pde': float(loss_pde.detach()),
            'loss_bc': float(loss_boundary.detach()),
            'rel_l2': relative_l2(model),
            'grad_norm': gradient_norm,
            'grad_cos_prev': gradient_cosine,
            'param_update_norm': update_norm,
            'effective_update': True,
            'elapsed_sec': time.perf_counter() - start_time,
        }
        history.append(row)
        print(
            f'[{optimizer}] step={step:5d} loss={row["loss"]:.4e} '
            f'relL2={row["rel_l2"]:.4e} grad_cos={row["grad_cos_prev"]:.4f}'
        )

result = {
    'stage': 2,
    'optimizer': optimizer,
    'seed': seed,
    'K': K,
    'n_pde': n_pde,
    'n_boundary_each': n_boundary_each,
    'lr': lr,
    'steps': steps,
    'schedule_steps': schedule_steps,
    'schedule_file': schedule_path.name,
    'device': str(device),
    'torch_version': torch.__version__,
    'history': history,
}

result_path = output_dir / output_name
result_path.write_text(json.dumps(result, indent=2))
print(f'Saved: {result_path}')

In [ ]:
print(f'Result file: {result_path}')
print(f'Device: {result["device"]}')
print(f'Final metrics: {result["history"][-1]}')

In [ ]:
import matplotlib.pyplot as plt

history = result['history']
steps_seen = [row['step'] for row in history]
rel_l2 = [row['rel_l2'] for row in history]
losses = [row['loss'] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(steps_seen, losses, marker='o')
axes[0].set_title('Training loss')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].grid(True, which='both', alpha=0.3)

axes[1].semilogy(steps_seen, rel_l2, marker='o')
axes[1].set_title('Relative L2 error')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Relative L2')
axes[1].grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import FileLink, display

display(FileLink(str(result_path)))